In [1]:
import pandas as pd
import numpy as np

from matminer.featurizers.structure import SiteStatsFingerprint, BagofBonds

# 1. Load the dataset
df = pd.read_pickle("perovskite_structures.pkl")

print(f"Loaded {len(df)} crystal structures.")

# Check structure column
if "structure" not in df.columns:
    raise ValueError("❌ 'structure' column not found in the dataset.")

# Remove missing structures
df = df.dropna(subset=["structure"]).reset_index(drop=True)

print(f"Valid structures: {len(df)}")

# 2. SiteStatsFingerprint
print("\nGenerating SiteStatsFingerprint features...")

ssf = SiteStatsFingerprint.from_preset(
    "CoordinationNumber_ward-prb-2017"
)

df_ssf = ssf.featurize_dataframe(
    df[["structure"]],
    col_id="structure",
    ignore_errors=True
)

ssf_features = df_ssf.drop(columns=["structure"], errors="ignore")

print(
    f"Generated {ssf_features.shape[1]} "
    "SiteStatsFingerprint features."
)

# 3. Bag of Bonds
print("\nGenerating Bag of Bonds features...")

bob = BagofBonds()

bob.fit(df["structure"].tolist())

df_bob = bob.featurize_dataframe(
    df[["structure"]],
    col_id="structure",
    ignore_errors=True
)

bob_features = df_bob.drop(columns=["structure"], errors="ignore")

print(
    f"Generated {bob_features.shape[1]} "
    "BagofBonds features."
)

# 4. Merge structural features
df_structural = pd.concat(
    [
        ssf_features.reset_index(drop=True),
        bob_features.reset_index(drop=True)
    ],
    axis=1
)

# Replace infinite values
df_structural = df_structural.replace(
    [np.inf, -np.inf],
    np.nan
)

# 5. Save
df_structural.to_csv(
    "structural_features_s22.csv",
    index=False
)

print("\n✅ Structural feature matrix successfully generated and saved.")
print("Final shape:", df_structural.shape)
print("File: structural_features_s22.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'perovskite_structures.pkl'